In [1]:
import pandas as pd
import os

# Folder path
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

files = {
    "users": "users.csv",
    "friends": "friends.csv",
    "user_games": "user_games.csv",
    "games": "games.csv"
}

dfs = {}

# -------------------
# LOAD FILES + ROW COUNTS
# -------------------
for name, file in files.items():
    path = os.path.join(data_path, file)
    df = pd.read_csv(path)
    dfs[name] = df

        # --- CLEANUP FOR USERS FILE ---
    if name == "users":
        # Remove useless column
        df.drop(columns=["last_logoff"], inplace=True, errors="ignore")

        # Convert account creation timestamp → datetime
        if "account_created" in df.columns:
            df["account_created"] = pd.to_datetime(
                df["account_created"],
                unit="s",
                errors="coerce"
            )

    dfs[name] = df
    print(f"\n===== {file} =====")
    print(df.head())
    print(f"{file}: {len(df):,} rows")


# -------------------
# DISCOVERED VS VISITED USERS
# -------------------
print("\n===== USER COUNTS =====")

# visited = users we actually crawled
visited_users = len(dfs["users"])
print(f"Visited users: {visited_users:,}")

# discovered ≈ anyone appearing in friend edges
friends_df = dfs["friends"]
discovered_users = pd.unique(
    pd.concat([friends_df["user_steamid"], friends_df["friend_steamid"]])
)

print(f"Discovered users (unique IDs in network): {len(discovered_users):,}")


# -------------------
# MISSING VALUES PER FILE
# -------------------
print("\n===== MISSING VALUES =====")
for name, df in dfs.items():
    print(f"\n{name}.csv")
    print(df.isna().sum())


# -------------------
# PUBLIC VS PRIVATE ACCOUNTS
# -------------------
print("\n===== ACCOUNT VISIBILITY =====")

users_df = dfs["users"]

if "public" in users_df.columns:
    public_counts = users_df["public"].value_counts(dropna=False)

    public = public_counts.get(True, 0)
    private = public_counts.get(False, 0)
    total = public + private

    print(f"Public accounts : {public:,} ({public/total:.2%})")
    print(f"Private accounts: {private:,} ({private/total:.2%})")
else:
    print("Column 'public' not found in users.csv")


===== users.csv =====
             steamid  public country     account_created
0  76561198064675174    True      IT 2012-06-02 17:21:29
1  76561198109425210    True      BE 2013-10-02 15:04:27
2  76561198281198045    True     NaN 2016-02-03 14:51:35
3  76561197994839969    True      BE 2007-12-12 16:48:56
4  76561197972380369    True      IT 2004-12-25 22:59:33
users.csv: 99,924 rows

===== friends.csv =====
        user_steamid     friend_steamid
0  76561198064675174  76561197967152992
1  76561198064675174  76561197968602157
2  76561198064675174  76561197970310193
3  76561198064675174  76561197970551918
4  76561198064675174  76561197970583217
friends.csv: 8,039,785 rows

===== user_games.csv =====
             steamid  appid  playtime_minutes
0  76561198064675174    400               367
1  76561198064675174    500               117
2  76561198064675174    550                 0
3  76561198064675174    620               710
4  76561198064675174   3900              7075
user_games.csv:

In [2]:
# Fill missing country values with "Unknown" just for clarity
users_df["country"] = users_df["country"].fillna("Unknown")

# Exclude "Unknown" for stats
known_countries = users_df[users_df["country"] != "Unknown"]

# Counts and percentages (excluding Unknown)
country_counts = known_countries["country"].value_counts()
country_percent = (known_countries["country"]
                   .value_counts(normalize=True)
                   .mul(100)
                   .round(2))

# Create dataframe
country_stats = pd.DataFrame({
    "count": country_counts,
    "percent": country_percent
})

# ------------------------
# Top 10 countries
# ------------------------
top_10_countries = country_stats.head(10)
print("=== Top 10 Countries ===")
display(top_10_countries)

# ------------------------
# Full list of countries ordered by count
# ------------------------
all_countries_sorted = country_stats.sort_values("count", ascending=False)
print("\n=== All Countries Sorted by Count ===")
display(all_countries_sorted)

=== Top 10 Countries ===


,count,percent
country,,
US,6527,11.22
CN,5060,8.70
BR,3915,6.73
RU,3511,6.04
DE,3040,5.23
PL,2230,3.83
GB,1851,3.18
TR,1780,3.06
FR,1709,2.94



=== All Countries Sorted by Count ===


,count,percent
country,,
US,6527,11.22
CN,5060,8.70
BR,3915,6.73
RU,3511,6.04
DE,3040,5.23
...,...,...
MW,2,0.00
MR,2,0.00
RE,2,0.00


In [3]:
# Exclude "Unknown"
known_countries = users_df[users_df["country"] != "Unknown"]

# List of countries ordered by count
all_countries_list = known_countries["country"].value_counts().index.tolist()

# Display in a single line
print(all_countries_list)

# Count how many users are from Italy
italy_count = (users_df["country"] == "IT").sum()  # Steam uses ISO codes
print(f"\nNumber of users from Italy: {italy_count}")

['US', 'CN', 'BR', 'RU', 'DE', 'PL', 'GB', 'TR', 'FR', 'PT', 'CA', 'JP', 'SE', 'IN', 'RO', 'DK', 'ID', 'NL', 'AU', 'CZ', 'AR', 'IT', 'NO', 'UA', 'ES', 'BE', 'FI', 'RS', 'PH', 'VN', 'SK', 'CH', 'MY', 'TW', 'HK', 'AT', 'MX', 'TH', 'KZ', 'HU', 'NE', 'KR', 'IL', 'PE', 'GR', 'SG', 'LT', 'IS', 'CL', 'BY', 'BG', 'AE', 'HR', 'ZA', 'NZ', 'PK', 'AQ', 'AF', 'EE', 'LV', 'CC', 'VA', 'CO', 'ZW', 'IE', 'CX', 'JM', 'KP', 'VE', 'BA', 'AL', 'IR', 'CG', 'UM', 'SA', 'MK', 'FJ', 'UG', 'AZ', 'SI', 'CU', 'CK', 'GE', 'UY', 'MT', 'PS', 'VI', 'IQ', 'GS', 'LB', 'HM', 'MN', 'CR', 'UZ', 'MC', 'EG', 'IM', 'DZ', 'SO', 'BD', 'TD', 'NG', 'MA', 'EC', 'QA', 'SY', 'CY', 'CD', 'JO', 'LU', 'LK', 'MD', 'TN', 'KG', 'PR', 'MG', 'ME', 'MM', 'XK', 'AM', 'FO', 'BS', 'CF', 'VG', 'AS', 'NP', 'GL', 'DO', 'DJ', 'MV', 'AO', 'KH', 'WF', 'SV', 'MZ', 'GT', 'KE', 'ET', 'KW', 'KY', 'TG', 'PG', 'MO', 'SM', 'AX', 'BN', 'BQ', 'BM', 'TJ', 'SC', 'PA', 'BF', 'KN', 'GH', 'OM', 'YE', 'BO', 'NF', 'LI', 'VC', 'ZM', 'MU', 'FM', 'PY', 'BW', 'NU', 'GU

US → United States

CN → China

BR → Brazil

RU → Russia

DE → Germany

PL → Poland

GB → United Kingdom

TR → Turkey

FR → France

PT → Portugal

CA → Canada

JP → Japan

SE → Sweden

IN → India

RO → Romania

DK → Denmark

ID → Indonesia

NL → Netherlands

AU → Australia

CZ → Czech Republic

AR → Argentina

IT → Italy

NO → Norway

UA → Ukraine

ES → Spain

BE → Belgium

FI → Finland

RS → Serbia

PH → Philippines

VN → Vietnam

SK → Slovakia

CH → Switzerland

MY → Malaysia

TW → Taiwan

HK → Hong Kong

AT → Austria

MX → Mexico

TH → Thailand

KZ → Kazakhstan

HU → Hungary

NE → Niger

KR → South Korea

IL → Israel

PE → Peru

GR → Greece

SG → Singapore

LT → Lithuania

IS → Iceland

CL → Chile

BY → Belarus

BG → Bulgaria

AE → United Arab Emirates

HR → Croatia

ZA → South Africa

NZ → New Zealand

PK → Pakistan

AQ → Antarctica

AF → Afghanistan

EE → Estonia

LV → Latvia

CC → Cocos (Keeling) Islands

VA → Vatican City

CO → Colombia

ZW → Zimbabwe

IE → Ireland

CX → Christmas Island

JM → Jamaica

KP → North Korea

VE → Venezuela

BA → Bosnia and Herzegovina

AL → Albania

IR → Iran

CG → Republic of the Congo

UM → United States Minor Outlying Islands

SA → Saudi Arabia

MK → North Macedonia

FJ → Fiji

UG → Uganda

AZ → Azerbaijan

SI → Slovenia

CU → Cuba

CK → Cook Islands

GE → Georgia

UY → Uruguay

MT → Malta

PS → Palestine

VI → U.S. Virgin Islands

IQ → Iraq

GS → South Georgia and South Sandwich Islands

LB → Lebanon

HM → Heard Island and McDonald Islands

MN → Mongolia

CR → Costa Rica

UZ → Uzbekistan

MC → Monaco

EG → Egypt

IM → Isle of Man

DZ → Algeria

SO → Somalia

BD → Bangladesh

TD → Chad

NG → Nigeria

MA → Morocco

EC → Ecuador

QA → Qatar

SY → Syria

CY → Cyprus

CD → Democratic Republic of the Congo

JO → Jordan

LU → Luxembourg

LK → Sri Lanka

MD → Moldova

TN → Tunisia

KG → Kyrgyzstan

PR → Puerto Rico

MG → Madagascar

ME → Montenegro

MM → Myanmar

XK → Kosovo

AM → Armenia

FO → Faroe Islands

BS → Bahamas

CF → Central African Republic

VG → British Virgin Islands

AS → American Samoa

NP → Nepal

GL → Greenland

DO → Dominican Republic

DJ → Djibouti

MV → Maldives

AO → Angola

KH → Cambodia

WF → Wallis and Futuna

SV → El Salvador

MZ → Mozambique

GT → Guatemala

KE → Kenya

ET → Ethiopia

KW → Kuwait

KY → Cayman Islands

TG → Togo

PG → Papua New Guinea

MO → Macau

SM → San Marino

AX → Åland Islands

BN → Brunei

BQ → Bonaire

BM → Bermuda

TJ → Tajikistan

SC → Seychelles

PA → Panama

BF → Burkina Faso

KN → Saint Kitts and Nevis

GH → Ghana

OM → Oman

YE → Yemen

BO → Bolivia

NF → Norfolk Island

LI → Liechtenstein

VC → Saint Vincent and the Grenadines

ZM → Zambia

MU → Mauritius

FM → Micronesia

PY → Paraguay

BW → Botswana

NU → Niue

GU → Guam

AG → Antigua and Barbuda

NI → Nicaragua

FK → Falkland Islands

HN → Honduras

TT → Trinidad and Tobago

BT → Bhutan

CW → Curaçao

TC → Turks and Caicos Islands

MF → Saint Martin

HT → Haiti

GI → Gibraltar

EH → Western Sahara

VU → Vanuatu

AD → Andorra

TM → Turkmenistan

LC → Saint Lucia

KI → Kiribati

PM → Saint Pierre and Miquelon

TK → Tokelau

LY → Libya

SL → Sierra Leone

CM → Cameroon

GP → Guadeloupe

TL → Timor-Leste

SZ → Eswatini

SD → Sudan

CI → Côte d’Ivoire

LR → Liberia

JE → Jersey

PN → Pitcairn Islands

GA → Gabon

LA → Laos

FX → Metropolitan France (old code, rarely used)

BZ → Belize

SJ → Svalbard and Jan Mayen

SH → Saint Helena

BB → Barbados

NR → Nauru

TZ → Tanzania

CV → Cape Verde

SS → South Sudan

AW → Aruba

BL → Saint Barthélemy

MP → Northern Mariana Islands

TO → Tonga

WS → Samoa

SX → Sint Maarten

BI → Burundi

NC → New Caledonia

SB → Solomon Islands

BH → Bahrain

PW → Palau

GG → Guernsey

DM → Dominica

IO → British Indian Ocean Territory

TF → French Southern and Antarctic Lands

KM → Comoros

YT → Mayotte

ER → Eritrea

MH → Marshall Islands

RW → Rwanda

ST → São Tomé and Príncipe

BV → Bouvet Island

AI → Anguilla

TV → Tuvalu

GQ → Equatorial Guinea

MS → Montserrat

GW → Guinea-Bissau

YU → Yugoslavia (old code, now replaced by RS/ME)

GM → Gambia

SN → Senegal

PF → French Polynesia

SR → Suriname

GN → Guinea

ML → Mali

GF → French Guiana

BJ → Benin

MW → Malawi

MR → Mauritania

RE → Réunion

MQ → Martinique

GD → Grenada

In [5]:
print("Oldest account:")
print(users_df["account_created"].min())

print("\nNewest account:")
print(users_df["account_created"].max())

Oldest account:
2003-09-11 01:29:16

Newest account:
2026-02-04 12:00:52


In [6]:
users_df["creation_year"] = users_df["account_created"].dt.year
users_df["creation_month"] = users_df["account_created"].dt.to_period("M")

year_counts = users_df["creation_year"].value_counts().sort_index()
month_counts = users_df["creation_month"].value_counts().sort_index()

year_counts

creation_year
2003.0      652
2004.0      923
2005.0      664
2006.0      760
2007.0     1197
2008.0     1403
2009.0     2160
2010.0     2585
2011.0     3731
2012.0     5488
2013.0     8210
2014.0     9921
2015.0    11209
2016.0     9484
2017.0     8295
2018.0     5293
2019.0     4568
2020.0     5085
2021.0     2944
2022.0     1959
2023.0     1515
2024.0     1022
2025.0      610
2026.0       22
Name: count, dtype: int64

In [7]:
# Ensure account_created is datetime
users_df["account_created"] = pd.to_datetime(users_df["account_created"], errors="coerce")

# Extract month number (1=Jan, 2=Feb, ..., 12=Dec)
users_df["creation_month_num"] = users_df["account_created"].dt.month

# Map month numbers to names
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
users_df["creation_month_name"] = users_df["creation_month_num"].map(month_names)

# Count users per month across all years
month_counts = users_df["creation_month_name"].value_counts().reindex(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
)

print("Account creations by month (all years combined):")
print(month_counts)

Account creations by month (all years combined):
creation_month_name
Jan    7925
Feb    6913
Mar    7259
Apr    6632
May    6516
Jun    7881
Jul    7767
Aug    7502
Sep    7037
Oct    7029
Nov    7645
Dec    9594
Name: count, dtype: int64


### Steam Friends Facts 
* base limit of friends: 250
* leveling Steam profile increases the cap by 5 friends for every level earned up to 2000
* friends are mutual

https://help.steampowered.com/en/faqs/view/1902-2D54-E445-59DD

In [13]:
# Number of unique users in user_games.csv
unique_users = user_games_df["steamid"].nunique()
print(f"Number of unique users in user_games.csv: {unique_users:,}")

Number of unique users in user_games.csv: 49,220


In [15]:
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"
user_games_df = pd.read_csv(os.path.join(data_path, "user_games.csv"))

# Convert minutes → hours
user_games_df["playtime_hours"] = user_games_df["playtime_minutes"] / 60

# -------------------
# Users with at least one unplayed game
# -------------------
users_with_unplayed = user_games_df[user_games_df["playtime_minutes"] == 0]["steamid"].nunique()
print(f"Number of users with at least one game they never played: {users_with_unplayed:,}")

# -------------------
# Games per user
# -------------------
games_per_user = user_games_df.groupby("steamid")["appid"].count()
print("\nGames per user:")
print(f"Min: {games_per_user.min()} games")
print(f"Median: {games_per_user.median():.0f} games")
print(f"Max: {games_per_user.max()} games")

# -------------------
# Total playtime per user
# -------------------
total_playtime_per_user = user_games_df.groupby("steamid")["playtime_hours"].sum()
print("\nTotal playtime per user (hours):")
print(f"Min: {total_playtime_per_user.min():,.2f} h")
print(f"Median: {total_playtime_per_user.median():,.2f} h")
print(f"Max: {total_playtime_per_user.max():,.2f} h")

# Top 10 users by total playtime
top_users = total_playtime_per_user.sort_values(ascending=False).head(10)
print("\nTop 10 users by total playtime (hours):")
for steamid, hours in top_users.items():
    print(f"{steamid}: {hours:,.2f} h")

# -------------------
# Total playtime per game
# -------------------
total_playtime_per_game = user_games_df.groupby("appid")["playtime_hours"].sum()
print("\nTotal playtime per game (hours):")
print(f"Min: {total_playtime_per_game.min():,.2f} h")
print(f"Median: {total_playtime_per_game.median():,.2f} h")
print(f"Max: {total_playtime_per_game.max():,.2f} h")

# Top 10 games by total playtime
top_games = total_playtime_per_game.sort_values(ascending=False).head(10)
print("\nTop 10 games by total playtime (hours):")
for appid, hours in top_games.items():
    print(f"{appid}: {hours:,.2f} h")

Number of users with at least one game they never played: 45,635

Games per user:
Min: 1 games
Median: 56 games
Max: 38420 games

Total playtime per user (hours):
Min: 0.00 h
Median: 2,988.62 h
Max: 2,266,794.75 h

Top 10 users by total playtime (hours):
76561198152959083: 2,266,794.75 h
76561198193401154: 2,265,379.78 h
76561198199186378: 1,408,437.80 h
76561198844100946: 1,232,288.97 h
76561198339649191: 998,464.75 h
76561198135380739: 723,297.28 h
76561198858160139: 676,861.88 h
76561198276335578: 637,090.22 h
76561197992597660: 576,246.62 h
76561198863249050: 544,268.52 h

Total playtime per game (hours):
Min: 0.00 h
Median: 26.55 h
Max: 54,895,210.95 h

Top 10 games by total playtime (hours):
730: 54,895,210.95 h
570: 14,431,474.32 h
578080: 5,876,129.02 h
271590: 5,177,553.10 h
252490: 5,049,801.07 h
440: 4,707,818.80 h
4000: 4,217,652.42 h
252950: 4,048,422.55 h
359550: 3,946,784.25 h
10: 2,994,765.62 h


In [20]:
friends_df = dfs["friends"]

# Unique users who have at least one friend recorded
unique_users = friends_df["user_steamid"].nunique()

print(f"Number of unique users in friends.csv (user_steamid): {unique_users:,}")

Number of unique users in friends.csv (user_steamid): 78,930


In [21]:
import pandas as pd
import networkx as nx

# Load friends dataframe
friends_df = dfs["friends"]

# -------------------
# 1. Basic counts
# -------------------
num_edges = len(friends_df)
unique_users = friends_df["user_steamid"].nunique()  # users we expanded
all_nodes = pd.unique(friends_df[["user_steamid", "friend_steamid"]].values.ravel())
num_nodes = len(all_nodes)

print(f"Number of nodes (unique users in edges): {num_nodes:,}")
print(f"Number of edges (friendships): {num_edges:,}")

# -------------------
# 2. Build NetworkX graph
# -------------------
G = nx.Graph()
# add all edges as undirected
edges = list(friends_df.itertuples(index=False, name=None))
G.add_edges_from(edges)

# -------------------
# 3. Degree statistics
# -------------------
degrees = [d for n, d in G.degree()]
avg_degree = sum(degrees) / len(degrees)
min_degree = min(degrees)
max_degree = max(degrees)
median_degree = int(pd.Series(degrees).median())

print("\nDegree statistics:")
print(f"Average degree: {avg_degree:.2f}")
print(f"Median degree: {median_degree}")
print(f"Min degree: {min_degree}")
print(f"Max degree: {max_degree}")

# -------------------
# 4. Density
# -------------------
density = nx.density(G)
print(f"\nNetwork density: {density:.6f}")

# -------------------
# 5. Connected components
# -------------------
components = list(nx.connected_components(G))
num_components = len(components)
largest_component_size = len(max(components, key=len))

print(f"\nNumber of connected components: {num_components}")
print(f"Largest connected component size: {largest_component_size:,}")

# -------------------
# 6. Average clustering coefficient
# -------------------
avg_clustering = nx.average_clustering(G)
print(f"\nAverage clustering coefficient: {avg_clustering:.4f}")

Number of nodes (unique users in edges): 6,084,524
Number of edges (friendships): 8,039,785

Degree statistics:
Average degree: 2.61
Median degree: 1
Min degree: 1
Max degree: 1999

Network density: 0.000000

Number of connected components: 1
Largest connected component size: 6,084,524

Average clustering coefficient: 0.0313


In [22]:
import networkx as nx

# -------------------
# Filter visited + public users
# -------------------
users_df = dfs["users"]
friends_df = dfs["friends"]

# Only public users
public_users = set(users_df.loc[users_df["public"], "steamid"])

# Keep edges where the source user is public
edges_public = [(u, f) for u, f in friends_df.itertuples(index=False, name=None) if u in public_users]

# Build NetworkX graph
G_pub = nx.Graph()
G_pub.add_edges_from(edges_public)

# -------------------
# Basic stats for public visited users
# -------------------
num_nodes_pub = G_pub.number_of_nodes()
num_edges_pub = G_pub.number_of_edges()

degrees_pub = [d for n, d in G_pub.degree()]
avg_degree_pub = sum(degrees_pub) / len(degrees_pub)
min_degree_pub = min(degrees_pub)
max_degree_pub = max(degrees_pub)
median_degree_pub = int(pd.Series(degrees_pub).median())

density_pub = nx.density(G_pub)

components_pub = list(nx.connected_components(G_pub))
num_components_pub = len(components_pub)
largest_component_size_pub = len(max(components_pub, key=len))

avg_clustering_pub = nx.average_clustering(G_pub)

# -------------------
# Print results
# -------------------
print(f"Number of nodes (visited & public): {num_nodes_pub:,}")
print(f"Number of edges: {num_edges_pub:,}")

print("\nDegree statistics:")
print(f"Average degree: {avg_degree_pub:.2f}")
print(f"Median degree: {median_degree_pub}")
print(f"Min degree: {min_degree_pub}")
print(f"Max degree: {max_degree_pub}")

print(f"\nNetwork density: {density_pub:.6f}")
print(f"Number of connected components: {num_components_pub}")
print(f"Largest connected component size: {largest_component_size_pub:,}")
print(f"Average clustering coefficient: {avg_clustering_pub:.4f}")

KeyboardInterrupt: 